In [15]:
import torch
from torch.nn import Module, ModuleList, Parameter, Buffer
import tiktoken
import math
import os
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
print('Hello World')

Hello World


## 1. Setting up the model

In [ ]:
class Linear(Module):
    def __init__(self, in_dim, out_dim):
        """ 
        Initialize a linear layer without a bias term.
        Inputs:
            in_dim : int - input feature dim
            out_dim : int - output feature dim
        """
        super().__init__()
        self.weight = Parameter(torch.randn(out_dim,in_dim)*math.sqrt(2/in_dim)) ### `He Initiation`
    def forward(self,X):
        """
        Apply linear layer to one or more input vectors.
        Input:
            X : torch.Tensor[float] (... x in_dim) - input tensor
        Output:
            torch.Tensor[float] (... x out_dim) - transformed tensor
        """
        return X@self.weight.T
    
class Embedding(Module):
    def __init__(self, num_tokens, dim):
        """
        Inputs:
            num_tokens : int - vocabulary size (i.e. how many token ids your model knows)
            dim : int - embedding dimention
        """
        super().__init__()
        ### weight is a matrix of embeddings for each of the V tokens (num_tokens)
        self.weight = Parameter(torch.randn(num_tokens,dim))
    def forward(self, Y):
        """
        Look up embeddings for an integer tensor of token ids
        Input:
            Y : torch.Tensor[int] (...) - tensor of token indices in [0, num_tokens)
        Output:
            torch.tensor[float] (... x dim) - embedding vectors for each token id
        """
        return self.weight[Y]

def silu(x):
    """
    Input:
        x : torch.Tensor[float] (...) - input tensor
    Output:
        torch.Tensor[float] (...) - tensor after applying SiLU
    """
    return x*torch.sigmoid(x)

def rms_norm(X, eps=1e-5):
    """
    Inputs:
        X : torch.Tensor[float] (... x dim) - input tensor
        eps: float - munerical stability constant
    Output:
        torch.Tensor[float] (... x dim) - RMS-normalized tensor
    """
    rms_X = torch.sqrt(torch.mean(X**2,dim=-1,keepdim=True)+eps)
    return X/rms_X

def self_attention(Q,K,V, mask=None):
    """
    Inputs:
        Q: torch.Tensor[float] (... x query_len x d) - query tensor
        K: torch.Tensor[float] (... x key_len x d) - key tensor
        V: torch.Tensor[float] (... x key_len x d_v) - value tensor
        mask: torch.Tensor[float] (... x query_len x key_len) or None - additive attention mask
    Output:
        torch.Tensor[float] (... x query_len x d_v) - attention output tensor
    """
    d = Q.shape[-1]
    scores = (Q@K.transpose(-1,-2))/math.sqrt(d)
    if mask is not None: ### masking stops mixing among past and future words.
        scores += mask
    return torch.softmax(scores, dim=-1)@V

class MultiHeadAttentionKVCache(Module):
    def __init__(self, dim, n_heads, max_cache_size):
        """
        Initialize a multi-head self-attention layer with KV cache buffers.
        
        Inputs:
            dim: int - total embedding dimention
            n_heads: int - number of attention heads
            max_cache_size: int - maximum sequence length stored in the cache
        """
        super().__init__()
        self.n_heads = n_heads
        self.each_head_dim = dim // n_heads
        assert dim % n_heads == 0

        self.wq = Linear(dim,dim)
        self.wk = Linear(dim,dim)
        self.wv = Linear(dim,dim)
        self.wp = Linear(dim,dim)
        
        ### KV cache (batch_size = 1)
        self.k_cache = Buffer(torch.zeros(1, max_cache_size, dim))
        self.v_cache = Buffer(torch.zeros(1, max_cache_size, dim))

    def forward(self, X, mask=None, seq_pos=0, use_kv_cache=False):
        """
        Apply multi-head self-attention, optionally updating and using the KV cache.

        Inputs:
            X: torch.Tensor[float] (batch_size x seq_len x dim) - input sequence embedding (this X is := self.weight[Y] after embedding)
            mask: torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos: int - starting sequence position for cached tokens
            use_kv_cache : bool - wheather to update and use the KV cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - attention output
        """
        B, N, D = X.shape # batch_size x seq_len (number of different elements/words in sequence/text) x embedding_dim
        Q, K, V = self.wq(X), self.wk(X), self.wv(X)

        if use_kv_cache:
            ### populate and update cache
            self.k_cache[:, seq_pos:seq_pos+N, :] = K
            self.v_cache[:, seq_pos:seq_pos+N, :] = V
            ###  prepare the entire updated cache to be used in self_attention()
            K = self.k_cache[:, :seq_pos+N, :]
            V = self.v_cache[:, :seq_pos+N, :]

        ### Multihead Attention: (B, N, D) => (B, N, n_heads, D/heads) = (B,N,self.n_heada, self.each_head_dim)
        Q = Q.reshape(B, N, self.n_heads, self.each_head_dim).transpose(1,2) # .transpose(1,2) as attention is computed per head.
        K = K.reshape(B, K.shape[1], self.n_heads, self.each_head_dim).transpose(1,2) #seq_len of K,V changes as we update cache (i.e. new len = .shape[1]) 
        V = V.reshape(B, V.shape[1], self.n_heads, self.each_head_dim).transpose(1,2) #seq_len of K,V changes as we update cache (i.e. new len = .shape[1])

        Y = self_attention(Q,K,V,mask)
        Y = Y.transpose(1, 2).reshape(B, N, D)
        return self.wp(Y)

class MLP(Module):
    def __init__(self, dim, ffn_dim):
        """
        Initialize the gated feed-forward network used in the transformer block.
        Inputs:
            dim : int - model dimension
            ffn_dim : int - hidden feed-forward dimension
        """
        super().__init__()
        self.w1 = Linear(dim,ffn_dim)
        self.w2 = Linear(ffn_dim,dim)
    def forward(self, X):
        """
        Apply the gated feed-forward network to the input tensor.
        Input:
            X : torch.Tensor[float] (... x dim) - input tensor
        Output:
            torch.Tensor[float] (... x dim) transformed tensor
        """
        return self.w2(silu(self.w1(X)))

class TransformerBlock(Module):
    def __init__(self, dim, n_heads, ffn_dim, max_cache_size):
        """
        Inputs:
            dim : int - model dimension
            n_heads : int - number of attention heads
            ffn_dim : int - hidden feed-forward dimension
            max_cache_size : int - maximum sequence length stored in the attention cache
        """
        super().__init__()
        self.attn = MultiHeadAttentionKVCache(dim,n_heads,max_cache_size)
        self.mlp = MLP(dim,ffn_dim)
    def forward(self, X, mask=None, seq_pos=0, use_kv_cacche=False):
        """
        Apply one trasformer block with residual connections.
        Inputs:
            X : torch.Tensor[float] (batch size x seq_len x dim) - input sequence embeddings
            mask : torch.Tensor[float] (seq_len x total_len) or None - additive attention mask
            seq_pos : int - starting sequence position for cached tokens
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x dim) - transformed sequence embeddings
        """
        z = X + self.attn(rms_norm(X),mask,seq_pos,use_kv_cacche)
        y = z + self.mlp(rms_norm(z))
        return y

class LLM(Module):
    def __init__(self, num_tokens, dim, n_heads, max_seq_len, ffn_dim, num_layers):
        """
        Inputs:
            num_tokens : int - vocabulary size (i.e. how many token ids your model knows)
            dim : int - model dimension
            n_heads : int - number of attention heads per layer
            max_seq_len : int - max supported sequence length (i.e. how many tokens model can process @ once)
            ffn_dim : int - hidden feed-forward dimension in each block
            num_layers : int - number of transformer blocks
        """
        super().__init__()
        ### Embedd your vocabulary
        self.embedding = Embedding(num_tokens,dim) 
        ### Fix the order invariance by position embedding
        self.pos_embedding = Parameter(torch.randn(max_seq_len,dim))
        ### a Pytorch Modulelist of length num_layers, where each element is a TransformerBlock
        self.layers = ModuleList()
        for i in range(num_layers):
            self.layers.append(TransformerBlock(dim,n_heads,ffn_dim,max_seq_len))
        self.output = Linear(dim,num_tokens)
        ### Fix past-future relation by masking
        mask = torch.triu(torch.full((max_seq_len, max_seq_len), float('-inf')), diagonal=1)
        self.mask = Buffer(mask)
    
    def forward(self, tokens, seq_pos=0, use_kv_cache=False):
        """
        Apply the full LLM to a batch of token sequences
        Inputs:
            tokens : torch.Tensor[int] (batch_size x seq_len) - input token ids (each row is a sentence)
            seq_pos : int - startig sequence position for positional embedding and cached attention
            use_kv_cache : bool - whether to update and use the attention cache
        Output:
            torch.Tensor[float] (batch_size x seq_len x num_tokens)
        """
        ### Embedd your input token ids
        X = self.embedding(tokens)
        N = X.shape[1]
        ### Add position embedding to your inputs
        X = X + self.pos_embedding[seq_pos:seq_pos+N]
        mask = self.mask[seq_pos:seq_pos+N, :seq_pos+N]
        ### Apply TransformerBlock for each layer
        for layer in self.layers:
            X = layer(X, mask, seq_pos, use_kv_cache)
        X = rms_norm(X)
        logits = self.output(X)
        return logits 

def cross_entropy_loss(logits, y):
    """
    Compute avg cross entropy loss over a minibatch
    Inputs:
        logits : ND torch.Tensor[float] (... x k) - predicted logits for each example
        y : (N-1)D torch.Tensor[int] (...) - desired class for each example
    Output:
        scalar torch.Tensor[float] - average cross entropy loss
    """
    logits_2D = logits.reshape(-1, logits.shape[-1])
    y_1D = y.reshape(-1)
    y_hat_y = torch.gather(logits_2D,1,y_1D.unsqueeze(1)).squeeze(1)
    loss = - y_hat_y + torch.logsumexp(logits_2D, dim=1)
    return loss.mean()

class Adam:
    def __init__(self, params, lr=1e-3, betas = (0.9, 0.999), eps=1e-8):
        """
        Initialize Adam optimizer state for a set of parameters.
        Inputs:
            params: iterable[torch.nn.parameter] - parameters to optimize
            lr: float - learning rate
            betas: tuple(float, float) - decay rates for first and second moments
            eps: float - numerical stability constant
        """
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps

        self.u = [torch.zeros(p.shape) for p in self.params]
        self.v = [torch.zeros(p.shape) for p in self.params]
        self.t = 0
    def step(self):
        """
        Apply one adam update to all stored parameter.
        """
        self.t += 1
        with torch.no_grad():
            for i,p in enumerate(self.params):
                if p.grad is not None:
                    self.u[i] = self.beta1 * self.u[i] + (1-self.beta1) * p.grad
                    self.v[i] = self.beta2 * self.v[i] + (1-self.beta2) * (p.grad**2)
                    u_hat = self.u[i] / (1-self.beta1**self.t)
                    v_hat = self.v[i] / (1-self.beta2**self.t)
                    p -= self.lr * u_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        """
        Zero out gradients for all stored parameters when gradients exist.
        """
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

## 2. Pretokenize a whole dataset using GPT-2 tokenizer. 

In [17]:
def pretokenize_data(tokenizer,in_filename,out_filename,chunk_size=2**20,max_chunks=None):
    """
    Inputs:
        tokenizer : object - tokenizer with an encode() method returning token ids
        in_filename : str - input text filename
        out_filename : str - output binary filename for unit16 tokens
        chunk_size : int - number of text characters to read at a time
        max_chunks : int or None - maximum number of chunks to tokenize
    Output:
        None - writes tokenized data to out_filename
    """
    with open(in_filename,'r') as fin, open(out_filename,'wb') as fout:
        chunk_count = 0
        while True:
            if max_chunks is not None and chunk_count >= max_chunks:
                break
            text = fin.read(chunk_size) ### How many text character to read in each chunk
            if not text:
                break
            tokens = tokenizer.encode(text, allowed_special='all')
            ### convert token ids to binary bytes
            fout.write(np.asarray(tokens, dtype=np.uint16).tobytes()) 
            chunk_count += 1
    return out_filename

In [18]:
from huggingface_hub import hf_hub_download

repo = "roneneldan/TinyStories"
filename = "TinyStoriesV2-GPT4-train.txt"
if not os.path.exists(filename):
    hf_hub_download(repo_id=repo,filename=filename,repo_type="dataset",local_dir=".")

In [19]:
tokenizer = tiktoken.get_encoding("gpt2") ### we use this gpt2 tokenizer for encoding
pretokenize_data(tokenizer,
                 "TinyStoriesV2-GPT4-train.txt",
                 "TinyStoriesV2-GPT4-train.small.bin",
                 chunk_size=2**20,
                 max_chunks=2)

'TinyStoriesV2-GPT4-train.small.bin'

#### ***3. Data loader for pretokenized chat conversations that returns token batches formatted for Transformer training.*** 

In [7]:
class DataLoader:
    def __init__(self,filename, seq_len, batch_size, device="cpu"):
        """
        Initialize a sequential token data loader backed by a binary file.
        Inputs:
            filename: str - binary filename containing unit16 token ids (i.e. out_filename from pretokenize_data)
            seq_len: int - number of tokens per input sequence (each row is a sequence, so how many columns in each row)
            batch_size: int - number of sequences per minibatch (how many row in each batch)
            device: str - device on which to place each minibatch tensor
        """
        self.filename = filename
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.device = device
        bytes_per_batch = batch_size * seq_len * 2
        file_size = os.path.getsize(filename)
        self.num_batches_tot = file_size // bytes_per_batch

    def __iter__(self):
        """
        Reset iteration state and return the iterator object.
        Output:
            DataLoader - iterate over token minimatches
        """
        self.current_batch = 0 ### Each time you start a new epoch, you need to go back to the beginning.
        return self
    
    def __next__(self):
        """
        Return the next input-target token minibatch.
        Output:
            tuple(torch.Tensor, torch.Tensor) - current input tokens and next-token targets
        """
        if self.current_batch >= self.num_batches_tot:
            raise StopIteration
        with open(self.filename, 'rb') as f: ### 'rb' => read binary
            f.seek(self.current_batch * self.seq_len * self.batch_size * 2)
            num_bytes = (self.seq_len + 1) * self.batch_size * 2 ### bytes to read
            raw_data = f.read(num_bytes)
            data = torch.tensor(np.frombuffer(raw_data,dtype=np.uint16).astype(np.int64))

        data = data.reshape(self.batch_size, self.seq_len+1) ### 1D (batch_size * (seq_len + 1) => 2D (batch_size, seq_len+1))
        data = data.to(self.device)

        X = data[:, :self.seq_len]
        Y = data[:, 1:] ### next seq_len tokens shifted by 1

        self.current_batch += 1

        return X, Y

#### ***4. Train Our LLM***
- Finally, let's put it all together. Train our LLM given a model, a data loader, and an optimizer. This sould now be a familiar loop:

1. Iterate over `x,y` pairs in the data loader (these will be the tokens and shifted token targets).

2. Apply the model to `x` and compute the cross entropy loss between the predicted outputs and `y`. One important elmenents to do here is to call `model(x).float()` (convert the output to a 32 bit floating point number, regardless of the model's native output), which will be very useful for accelerated training using lower precision.
3. Call the normal form of the update:

    opt.zero_grad()

    loss.backward()

    opt.step()

Repeat this over the whole data loader. Given that this can take some time for the larger models, it would be good to e.g., print the number of tokens seen and the loss achieved over the iterations.

In [8]:
def train_llm(model, loader, opt):
    """
    Run one full training pass over a token data loader.
    Inputs: 
        model: Module - language model mapping token sequences to logits
        loader: iterable - yields minibatches of input tokens and next-token targets
        opt: optimizer - object with zero_grad() and step() methods
    Output:
        None: updates model parameters in place
    """
    for i , (X,Y) in enumerate(loader):
        logits = model(X).float()
        loss = cross_entropy_loss(logits,Y)
        if opt is not None:
            opt.zero_grad()
            loss.backward()
            opt.step()

        if i % 10 == 0:
            print(f"iter {i}, loss {loss.item():.4f}") ### .items() converts tensor → plain number

In [9]:
### Train a very small LLM
loader = DataLoader("TinyStoriesV2-GPT4-train.small.bin", 512, 16, device="cpu") #filename, seq_len, batch_size, device => returns X,Y
# number of batches (63) = size(TinyStoriesV2-GPT4-train.small.bin)//(512*16*2)

model = LLM(num_tokens=tokenizer.n_vocab,
            dim=256,
            n_heads=8,
            max_seq_len=512,
            ffn_dim=512,
            num_layers=4)

opt = Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.95))
train_llm(model, loader, opt) #model, loader, opt => print(iteration, loss)

iter 0, loss 11.8976
iter 10, loss 7.4464
iter 20, loss 6.0586
iter 30, loss 5.7819
iter 40, loss 5.6047
iter 50, loss 5.5621
iter 60, loss 5.2474


#### ***5. Generation***

Since our model is trained, let's generate responses from the LLM sequentially by autoregressively sampling tokens given an input prompt.
  

In [12]:
def generate(model, prompt_tokens, tokenizer, temp=0.7, max_tokens=500, verbose=True):
    """
    Autoregressively sample tokens from a language model using its KV cache.
    Inputs:
        model: Module - language model mapping token sequences to logits
        prompt_tokens: list[int] - initial prompt tokens
        tokenizer: object - tokenizer with decode() and stop_tokens
        temp: float - sampling temperature
        max_tokens: int - max number of new tokens to generate
        verbose: bool - whether to print each generated token as it is sampled
    Output:
        list[int] - generated tokens, excluding the prompt tokens
    """
    device = next(model.parameters()).device
    ### Our model expect tokens in torch.Tensor[int] (batch_size x seq_len) and .reshape(1,-1) add that batch_size - input token ids
    tokens = torch.tensor(prompt_tokens, device=device).reshape(1,-1) ### ensures your model and input token tensor set to the same device (CPU or GPU) 
    logits = model(tokens, seq_pos=0, use_kv_cache=True) ### logits.shape => (batch_size, seq_len, num_tokens) #tokens start from position 0
    seq_pos = tokens.shape[1] #now update the seq_pos i.e. go to the last position of seq_len and start from there

    output = [] ### This will not include prompt tokens, only the generated tokens.

    ### for each batch size the model will go through each sequence (row) and generete new tokens (columns) which will append then to each row.
    ### before, (B,T,V) => (B,T,V+new_generated_tokens)
    for _ in range(max_tokens):
        ### Temperature sampling: P = softmax(logits_T), logits_T - the last row of the logits
        prob = torch.softmax(logits[:,-1,:]/temp, dim=-1) ### for all the batches take the last seq
        ### now let's generate tensor samples which belongs to this probability distribution 
        next_tokens = torch.multinomial(prob, 1) 
        token_id = next_tokens.item() # Returns the value of this tensor as a standard Python number
        output.append(token_id)

        if verbose:
            print(tokenizer.decode([token_id]), end="", flush=True)
        if token_id == tokenizer.eot_token: # you will see this: "<|endoftext|>", once generating text is done.
            break

        logits = model(next_tokens, seq_pos, use_kv_cache=True) # we only pass the new token not the entire token
        ### Y_T+1 = LLM(X_T+1)
        seq_pos += 1 # Now again the seq_pos will start internally from tokens.shape[1]+1, ...then again tokens.shape[1]+1+1
    
    return output

In [14]:
### encode text to token ids and pass the token ids (int) into generate()
prompt = tokenizer.encode("Once upon a time,") #this tokenizer is used (179 cell): tiktoken.get_encoding("gpt2") 
print("Once upon a time,")
generate(model, prompt, tokenizer);
# because of this decode line in generate: "print(tokenizer.decode([token_id]), end="", flush=True)" it will generate texts.

Once upon a time,
 go,

.
 day, there was had in, "." Tim they said, and are, "I a."
ie the to play snow. They?" The bug and Ben with his said, will. He long doll Tom to the truck. It it. They be happy and said, but metal, and Ben. It and Sue that he had saw a small Tom felt grape that was very with felt, a time, and up.
 able. They loved to dog. He was very was a time. They " day, there was they wereue. He create be Lily the town. It, friends, "I proud the cat, there was a big named The boy named
<|endoftext|>

## Brick by brick, we assembled every component of an LLM and now see the full system generating text seamlessly. 

# `Hurray !!!` 